In [ ]:
%matplotlib inline
!pip install opencv-python
!pip install PyMuPDF
!pip install matplotlib
!pip install pypdf
import matplotlib.pyplot as plt
import os
import cv2
import numpy as np
import fitz
from pypdf import PdfReader, PdfWriter
import time


In [ ]:
import sys
sys.path.append('../src')

from pdf_set import PDFSet

In [ ]:
data = PDFSet.fromFolder('../case_outputs/yellow', 'fitz numpy300 ')

In [ ]:
#### Follows grayscale -> blur -> threshold -> contour

pdf, img, fn = data[0,0]
img = img.copy()

def rect_bound(approx) -> tuple[np.ndarray, int]:
    x, y, w, h = cv2.boundingRect(approx)
    box = np.array([
        [x, y],
        [x + w, y],
        [x + w, y + h],
        [x, y + h]
    ])
    return box

def para_bound(approx) -> tuple[np.ndarray, int]:
    rect = cv2.minAreaRect(approx)  # note: use the original contour, not approx
    box = cv2.boxPoints(rect)
    box = np.intp(box)
    return box


# Reduce image channels
gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY) if img.ndim == 3 else img

# Reduce image noise
blur = cv2.GaussianBlur(gray, (5, 5), 0) 

# Define adaptive threshold to convert image to BW, adjust for light
thresh = cv2.adaptiveThreshold(blur, 255, cv2.ADAPTIVE_THRESH_MEAN_C,
                               cv2.THRESH_BINARY_INV, 15, 10)

# Find external contours, and compress information
contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

boxes = []
for contour in contours:
    # Simplify contour shape to retrieve bounding polygon
    approx = cv2.approxPolyDP(contour, 0.02 * cv2.arcLength(contour, True), True)
    if len(approx) == 4 and cv2.isContourConvex(approx):
        box = rect_bound(approx)
        cv2.drawContours(img, [box], 0, (0, 255, 0), 3)

plt.figure(figsize=(5,5))
plt.imshow(img)
plt.axis("off")
plt.title("Detected Boxes")
plt.show()
